### Setting the topology

In a YAML file, define the parameters and topology for your test in a similar manner as the following:
```yaml
topology:
  name: "g5k_mcast_eval"
  wall_time: "2hr"
  relay_nodes: false # whether to add one relay machine in each cluster
  netns_per_client: 5 # number of network namespaces to run on each client
  # see the possible frrouting version at https://deb.frrouting.org/
  frrouting_version: "frr-10.4"
  router_template: "base_router_config_ospf.frr" # path to the router configuration template

  server:
    cluster: "chirop" # Lille
    nodes: 1
    # node: "chirop-5.lille.grid5000.fr"   # optional: pin a specific machine

  # each site has one router + num_clients clients and one relay,
  # all reserved in the given cluster. 
  sites:
    - name: nancy
      cluster: gros
      num_clients: 5
    - name: rennes
      cluster: parasilo
      num_clients: 5
    - name: nantes
      cluster: ecotype
      num_clients: 5
    - name: lyon
      cluster: nova
      num_clients: 5

  # links are established between routers in different clusters
  # GRE tunnels are established between the two routers, with OSPF running over it.
  # endpoints must be router_server or router_client_CLIENT-CLUSTER-ID
  links:
    - [router_server, router_client_0]             # src -> nancy
    - [router_client_0, router_client_1]           # nancy -> rennes
    - [router_client_0, router_client_3]           # nancy -> lyon
    - [router_client_0, router_client_2]           # rennes -> nantes
``` 


In [ ]:
# !pip install enoslib ipywidgets==8.1.5 fabric --break-system-packages
!pip install -U jupyterlab ipywidgets jupyterlab-widgets --break-system-packages


### Setting up the experiment
Once you have your `topology.yaml` file, you can create the `G5KExpe` class which will handle most things for you.

In [1]:
from g5k_eval import G5KExpe

experiment = G5KExpe(
    # change the path to point to your topology yaml file
    topology_conf="./mcast_eval.yaml",
    #
    # other parameters exist:
    # g5k_conf_file_loc points to your .python-grid5000.yaml file which contains your grid5000 credentials, by default it is in `~/` (so `/home/USERNAME`)
    # g5k_conf_file_loc=".python-grid5000.yaml"
    #
    # job_type should be deploy, but you may need it to be different
    # job_type="deploy"
    #
    # os_env_name defines the OS environement that is deployed on the machines
    # by default it is debian12 with NFS, however you can find the entire list at https://www.grid5000.fr/w/Getting_Started#:~:text=On%20Grid%275000%20reference%20environments
    # Make sure to pick debian to ensure that the packages are properly installed
    # os_env_name="debian12-nfs"
    #
    # You can configure the number of ansible forks used, ansible's default is 5, meaning that it'll run commands on at most 5 host at once
    # in this framework the default is 25 to make use of more parallelism, however, increasing this value will consume more resources (especially memory)
    # setting the number of forks to 200 will consume around 25 GB of memory but will allow ansible to perform operations on 200 hosts at the same time
    ansible_forks=20,
)

# you should always follow grid5000's usage policy (see https://www.grid5000.fr/w/Grid5000:UsagePolicy)
# this method simply checks that the job you are trying to start will not cross the day-night boundary.
# If it does, it'll warn you. You can always comment this out if you wish...
experiment.usage_policy_check()

provider = experiment.setup_enoslib_conf()

_____        ___  ____  _ _ _
 | ____|_ __  / _ \/ ___|| (_) |__
 |  _| | '_ \| | | \___ \| | | '_ \
 | |___| | | | |_| |___) | | | |_) |
 |_____|_| |_|\___/|____/|_|_|_.__/  10.9.0

 • Documentation: ]8;id=886334;https://discovery.gitlabpages.inria.fr/enoslib/\https://discovery.gitlabpages.inria.fr/enoslib/]8;;\                            
 • Source: ]8;id=521910;https://gitlab.inria.fr/discovery/enoslib\https://gitlab.inria.fr/discovery/enoslib]8;;\                                         
 • Chat: ]8;id=189945;https://framateam.org/enoslib\https://framateam.org/enoslib]8;;\

                         Dependency check                         
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Provider      ┃    Status     ┃ Hint                           ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Chameleon     │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ ChameleonKVM  │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ ChameleonEdge │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ Fabric        │ NOT INSTALLED │ pip install enoslib[fabric]    │
│ Distem        │ NOT INSTALLED │ pip install enoslib[distem]    │
│ IOT-lab       │ NOT INSTALLED │ pip install enoslib[iotlab]    │
│ Grid'5000     │   INSTALLED   │                                │
│ Openstack     │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ Vagrant       │ NOT INSTALLED │ pip install enoslib[vagrant]   │
│ VMonG5k       │   INSTALLED   │                                │
└───────────────┴───────────────┴────────────────────────────────┘

                                Connectivity check                                 
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Provider  ┃ Key                 ┃ Connectivity ┃ Hint                           ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Grid'5000 │ ssh:access          │      ✅      │ Connection to access.grid5000… │
│ Grid'5000 │ ssh:access:frontend │      ✅      │ Connection Host(rennes.grid50… │
│ Grid'5000 │ api:access          │      ✅      │                                │
│ VMonG5k   │ access              │      ❔      │ Check G5k status               │
└───────────┴─────────────────────┴──────────────┴────────────────────────────────┘


### Reserving resources
Now that G5K is setup, we can create the experiment's reservation by defining the number of machines of each role and in each cluster.

Once done, we proceed with the actual reservation of the machines. Be aware that this step may take some time (minimum 5 minutes). This is due to the deployment of the VM image. 

**Don't forget to run "ssh-add KEY_PATH" to allow ansible to connect using your ssh key**

In [ ]:
!pip install -U jupyterlab ipywidgets jupyterlab-widgets --break-system-packages
experiment.reserve_res(provider)
display(experiment.roles)

Defaulting to user installation because normal site-packages is not writeable
Reserving resources now, might take a while...


INFO     [ProviderS] Common reservation_date=2026-09-18T09:42:22 (local time) ]8;id=186597;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/providers.py\providers.py]8;;\:]8;id=46934;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/providers.py#60\60]8;;\
         [5 providers]                                                                       

INFO     [G5k] Submitting {'name': 'g5k_mcast_eval', 'types': ['deploy', ]8;id=642823;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=764836;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#306\306]8;;\
         'origin=enoslib_g5k'], 'resources': "{cluster='chirop'}/nodes=1                     
         +{cluster='chirop'}/nodes=1+slash_22=1,walltime=4:00:00",                           
         'command': 'sleep 31536000', 'queue': 'default', 'reservation':                     
         '2026-09-18 09:42:23'} on lille                                                     

INFO     [G5k] Submitting {'name': 'g5k_mcast_eval', 'types': ['deploy', ]8;id=505654;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=914372;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#306\306]8;;\
         'origin=enoslib_g5k'], 'resources': "{cluster='ecotype'}/nodes=                     
         1+{cluster='ecotype'}/nodes=5+slash_22=1,walltime=4:00:00",                         
         'command': 'sleep 31536000', 'queue': 'default', 'reservation':                     
         '2026-09-18 09:42:28'} on nantes                                                    

INFO     [G5k] Submitting {'name': 'g5k_mcast_eval', 'types': ['deploy', ]8;id=591329;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=29145;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#306\306]8;;\
         'origin=enoslib_g5k'], 'resources': "{cluster='nova'}/nodes=1+{                     
         cluster='nova'}/nodes=5+slash_22=1,walltime=4:00:00",                               
         'command': 'sleep 31536000', 'queue': 'default', 'reservation':                     
         '2026-09-18 09:42:30'} on lyon                                                      

...

INFO     [G5k] Job 338265 on nantes: scheduled for 2026-09-18 09:42:29   ]8;id=973209;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=531085;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#347\347]8;;\

INFO     [G5k] Job 4117184 on rennes: scheduled for 2026-09-18 09:45:22  ]8;id=159211;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=793319;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#347\347]8;;\

INFO     [G5k] All jobs are Running !                                    ]8;id=178322;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=751744;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#358\358]8;;\

INFO     [G5k] Checking environment on reloaded nodes                         ]8;id=705533;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/provider.py\provider.py]8;;\:]8;id=653669;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/provider.py#745\745]8;;\

Output()

**Optional**: You can refresh the roles by running the cell below (useful when the notebook closed but you have a reservation running)

In [ ]:
experiment.sync_info()

#### Setting up interfaces, IP subnets, and Network namespaces

In [ ]:
experiment.setup_interfaces()
experiment.assign_node_ips()
experiment.netns_setup_macvlan()

### Setting up GRE tunnels between routers in different clusters

This step will create GRE tunnels between each pair of routers as defined in the topology file. The endpoints of the tunnels use the production IP of the nodes.

In [ ]:
experiment.setup_gre_tunnels()

#### FRRouting setup

With GRE tunnels setup between routers, we can now configure and start FRRouting. The frr configuration template defined in the topology file will be used as a base.

In [ ]:
experiment.frrouting_setup()
experiment.setup_default_routes()

### Upload binary files over to nodes

We build the executables locally first

In [ ]:
# TODO: change this path to your project
!cd ../../../g5k_mcast_eval && cargo build --release

Then we push them to the nodes

In [ ]:
import enoslib as en

experiment.push_binaries(
    # TODO: change these paths with the path to your binaries and certificates
    bin_dir="../../../g5k_mcast_eval/target/release",
    cert_dir="../../../g5k_mcast_eval",
    relay_binaries=(),
)

res = en.run_command(
    "sysctl -w net.core.rmem_default=26214400 && sysctl -w net.core.rmem_max=26214400",
    roles=experiment.roles,
)
print("errors: " + str([out.stderr for out in res.filter(status=en.STATUS_FAILED)]))

### Running the relay experiment

Experiment.py provides some basic blocks that should (ideally) allow you to define your own custom experiments.
Below you will find the code for the evaluation of two types of Flexicast QUIC relays, this should hopefully provide enough information.


In [ ]:
import concurrent.futures
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Literal
import time

import enoslib as en

from g5k_eval.experiment import (
    EvalConfig,
    MetricSpec,
    collect_results,
    run_eval,
)
from g5k_eval.remote import (
    run_cmd_bg_enos,
    run_cmd_ssh_parallel,
    send_pkill_hosts,
    ssh_bg_hosts,
)


@dataclass
class RunConfig:
    additional_data_size: int
    test_length: int


@dataclass
class CatEvalConfig(EvalConfig):
    ready_sleep_clients: int = 2
    post_test_buffer: int = 7
    bin_log_level: str = "info"
    cert_path: str = "/tmp"
    server_bin: str = "/tmp/bin/server"
    client_bin: str = "/tmp/bin/client"
    remote_log_root: str = "/tmp/logs"
    num_ns_per_client: int = experiment.topology.netns_per_client
    cc_algo: str = "cubic"
    fallback_delay: int = 10000
    server_cpus: str = "0-7"  # taskset -c range for server
    per_cluster_results: bool = True
    flow_control: int = 16_000_000  # 16 Megabytes


# define the results to extract from the client logs, one csv output file is emitted for each metric
METRICS = [
    MetricSpec(
        key="LATENCY",
        column="y_LATENCY",
        pattern=rf"^RESULT-LATENCY-\S+\s+([0-9.]+)\s*$",
    ),
    #  you can add more result types here, e.g.:
    # MetricSpec(key="THROUGHPUT", column="y_THROUGHPUT"),
]

We can now define specific tests based on our `RunConfig`.

For the network categorization test, we simply send one packet of increasing size, and we wait until it has been received by all clients.
- 1KB will fit inside of one packet
- 10KB will fit in one flight of packets
- for larger values, the sender will have to grow its congestion window to be able to send it

In [ ]:
def categorization_matrix():
    return [
        RunConfig(additional_data_size=sz, test_length=length)
        # for sz in (10_000, 100_000, 1_000_000, 10_000_000)
        # for sz in (1_000,)
        # for sz in (1_000_000,)
        # for sz in (10_000_000,)
        # for sz, length in ((10_000_000, 20),)
        for sz, length in (
            # (1_000, 10),
            (10_000, 15),
            (100_000, 15),
            (1_000_000, 20),
            (10_000_000, 30),
        )
    ]

To enable us to have graphs that show certain metrics per cluster, we need to pass in a list of the cluster names and their subnets to the clients. Here we construct the lists to pass to the clients

In [ ]:
import ipaddress
import pickle
from pathlib import Path

TOPOLOGY_CACHE = Path("./topology_cache.pkl")


# ---------- cluster names & subnets ----------
# the subnets are in experiment.networks, but i need the subnets keyed by their index and not their cluster
def compute_topology_cache():
    site_subnets = {
        site.name: {
            "cluster": site.cluster,
            "num_clients": site.num_clients,
            # NOTE: we only remove the prefix because the arg is an IPv4Addr (in rust), not a network  # "10.x.y.0"
            "subnet": str(experiment.networks[f"subnet_client_{i}"][0].network),
        }
        for i, site in enumerate(experiment.topology.sites)
    }
    return {
        "server_subnet": str(experiment.networks["subnet_server"][0].network),
        "site_subnets": site_subnets,
        "cluster_names": [site.name for site in experiment.topology.sites],
        "cluster_subnets": [
            str(ipaddress.ip_network(info["subnet"]).network_address)
            for info in site_subnets.values()
        ],
        "site_subnets_str": [info["subnet"] for info in site_subnets.values()],
    }


try:
    cache = compute_topology_cache()
    with open(TOPOLOGY_CACHE, "wb") as f:
        pickle.dump(cache, f)
    print(f"saved topology values to {TOPOLOGY_CACHE}")
except (NameError, AttributeError, KeyError):
    print(f"no running experiment, loading topology values from {TOPOLOGY_CACHE}")
    with open(TOPOLOGY_CACHE, "rb") as f:
        cache = pickle.load(f)

server_subnet = cache["server_subnet"]
site_subnets = cache[
    "site_subnets"
]  # NOTE: "subnet" is now a string, e.g. "10.x.y.0/22"
cluster_names = cache["cluster_names"]
cluster_subnets = cache["cluster_subnets"]
site_subnets_str = cache["site_subnets_str"]

print(f"site_subnets_str: {site_subnets_str}")
print(f"cluster_names:  {cluster_names}")
print(f"cluster_subnets: {cluster_subnets}")

Now that the experiment is defined, we need to specify the commands that will be ran on the nodes.


In [ ]:
# ---------- command builders ----------
def server_cmd(cfg, rc, server_ip, run_dir, sleep_deadline_ts):
    length = rc.test_length * 2
    qlog = f"{run_dir}/qlog/server"
    return (
        f"mkdir -p {qlog} && "
        f"env QLOGDIR={qlog} RUST_LOG_STYLE=never RUST_BACKTRACE=full "
        f"RUST_LOG={cfg.bin_log_level} taskset -c {cfg.server_cpus} {cfg.server_bin} "
        f"--cert-path {cfg.cert_path} --src {server_ip}:4433 --mc-src-addr {server_ip}:4443 "
        f"--flexicast --fc-timer 0 --fall-back-delay {cfg.fallback_delay} "
        f"--unicast --fec-scheduler noredundancy --length {length} "
        f"--cc-algorithm {cfg.cc_algo} --fc-cwnd {cfg.cc_algo} "
        f"--additional-data-size {rc.additional_data_size} --test-start-ts {sleep_deadline_ts} "
        f"--initial-fc-flow {cfg.flow_control} "
    )


# IMPORTANT NOTE: since we have multiple network namespaces defined on each client machine,
# we can run processes in these namespaces using the naming scheme "client-$NS_IDX" (with NS_IDX going from the number of 0 to NSs)
def client_loop_cmd(cfg, rc, server_ip, run_dir, node_id, sleep_deadline_ts):
    qlog_base = f"{run_dir}/qlog/client"
    per_cluster_res = "--per-cluster-results" if cfg.per_cluster_results else ""
    return f"""
mkdir -p {run_dir}/client
pids=()
for NS_IDX in $(seq $(( {cfg.num_ns_per_client} - 1 )) -1 0); do
    GLOBAL_IDX=$(( {node_id} * {cfg.num_ns_per_client} + NS_IDX ))
    CLIENT_ID=$(( GLOBAL_IDX + 1 ))
    NS_NAME="client-$NS_IDX"
    mkdir -p {qlog_base}_$CLIENT_ID
    CLIENT_IP=$(ip netns exec $NS_NAME ip -f inet addr show | grep inet | tail -1 | awk '{{print $2}}' | cut -d'/' -f1)
    ip netns exec $NS_NAME env QLOGDIR={qlog_base}_$CLIENT_ID RUST_LOG_STYLE=never RUST_BACKTRACE=full RUST_LOG={cfg.bin_log_level} \\
        {cfg.client_bin} --server-ip {server_ip} --port 4433 \\
        -l $CLIENT_IP --flexicast -u CLIENT$CLIENT_ID --length {rc.test_length} \\
        --test-start-ts {sleep_deadline_ts} \\
        --additional-data-size {rc.additional_data_size} --cc-algorithm {cfg.cc_algo} --flow-control {cfg.flow_control}  \\
        {per_cluster_res} \\
         {" ".join(f"--cluster-names={name}" for name in cluster_names)} \\
         {" ".join(f"--cluster-subnets={subnet}" for subnet in cluster_subnets)} \\
        > {run_dir}/client/client_$CLIENT_ID.stdout \\
        2> {run_dir}/client/client_$CLIENT_ID.stderr < /dev/null < /dev/null &
    pids+=($!)
done
for pid in "${{pids[@]}}"; do wait $pid; done
"""


# ---------- one run of the relay experiment ----------
def run_once(cfg, rc, run_index, test_name):
    """Run one iteration: start the server, start the clients in
    their namespaces, wait for the test to finish, then collect the results."""
    roles_dict = experiment.roles
    node_ips = experiment.node_ips

    server_ip = node_ips["server"][0]
    # NOTE: very important, make sure that this run_id is the same as the one in the SQLOG collection cell below
    run_id = f"sz{rc.additional_data_size}_r{run_index}"
    run_dir = f"{cfg.remote_log_root}/{test_name}/{run_id}"

    # make sure that each client is root because it has to start the clients in network namespaces
    client_hosts = [
        en.Host(h.address, alias=h.alias, user="root", extra=h.extra)
        for h in roles_dict["client"]
    ]
    all_hosts = roles_dict["server"] + client_hosts

    # create the dirs on all of the hosts and stop anything left over from a
    # previous run
    run_cmd_ssh_parallel(
        f"mkdir -p {run_dir}/server {run_dir}/client {run_dir}/qlog ; "
        f"pkill -9 server || true ; pkill -9 client || true",
        all_hosts,
    )
    time.sleep(1)

    # pick a timestamp in 5 seconds, we pass this to all of the clients that will all wait until that timestamp is reached before starting
    datetime_now = datetime.now()
    sleep_deadline = datetime_now + timedelta(seconds=5)
    sleep_deadline_ts = sleep_deadline.timestamp()

    # start server in bg
    run_cmd_bg_enos(
        server_cmd(cfg, rc, server_ip, run_dir, sleep_deadline_ts),
        roles_dict["server"],
        stdout=f"{run_dir}/server/server.stdout",
        stderr=f"{run_dir}/server/server.stderr",
        task_name="server",
    )

    # start all clients at once, in a single ansible run: each host gets its own
    # command, and they all wait for sleep_deadline_ts so they start together
    ssh_bg_hosts(
        [
            (
                h,
                client_loop_cmd(
                    cfg, rc, server_ip, run_dir, node_id, sleep_deadline_ts
                ),
                f"{run_dir}/client/loop_{node_id}.stdout",
                f"{run_dir}/client/loop_{node_id}.stderr",
            )
            for node_id, h in enumerate(client_hosts)
        ],
        task_name="clients",
    )
    print("Started clients")

    # wait for test duration to pass
    time.sleep(rc.test_length + cfg.post_test_buffer)

    send_pkill_hosts(all_hosts, ["server", "client"])

    time.sleep(1)

    results = collect_results(cfg, client_hosts, run_dir, test_name, METRICS)
    return results, []

Lauching the test

In [ ]:
import logging

logging.getLogger("paramiko").setLevel(logging.WARNING)

N_RUNS = 10

cfg = CatEvalConfig(n_runs=N_RUNS, monitor_cpu=False)
now = datetime.now().strftime("%d-%m-%H-%M%p")

test_name = f"categorization_{now}"
matrix = categorization_matrix()


def row_fields(rc):
    # columns identifying each run in the result CSVs
    return {
        "ADDITIONAL_DATA_SIZE": rc.additional_data_size,
    }


run_eval(
    matrix,
    cfg,
    run_once=run_once,
    test_name=test_name,
    row_fields=row_fields,
    metrics=METRICS,
)

### Downloading SQLOGs from server and relay

In [ ]:
import shutil
import subprocess
from pathlib import Path

# uses cfg and test_name from the launching cell above
local_base = Path(f"./sqlogs/{test_name}")
local_base.mkdir(parents=True, exist_ok=True)

client_hosts = [
    en.Host(h.address, alias=h.alias, user="root", extra=h.extra)
    for h in experiment.roles["client"]
]

# for each client machine, download all of the qlogs from the namespaces that are stored in tmp (/tmp in g5k machines is stored on a local disk)
for client_host in client_hosts:
    host = client_host.address
    print(f"downloading sqlogs from {host}")
    subprocess.run(
        [
            "rsync",
            "-az",
            "--include=*/",
            "--include=*.sqlog",
            "--exclude=*",
            "--prune-empty-dirs",
            f"root@{host}:{cfg.remote_log_root}/{test_name}/",
            f"{local_base}/",
        ],
        check=False,
    )

# after we downloaded the files, we need to move the files up one level to remove the "qlog" dir
# ./sqlogs/{test_name}/{run_id}/client_{CLIENT_ID}
for qlog_dir in sorted(local_base.glob("*/qlog")):
    for client_dir in sorted(qlog_dir.iterdir()):
        target = qlog_dir.parent / client_dir.name
        # if the same results were already downloaded before, we replace the prev copy
        if target.exists():
            shutil.rmtree(target)
        client_dir.rename(target)
    qlog_dir.rmdir()

print(f"results: {local_base}")

### Merging SQLOG files together

In [ ]:
from pathlib import Path
import sys
import re
import csv
import ipaddress
import multiprocessing as mp
import orjson


site_subnet_to_cluster = {
    ipaddress.ip_network(subnet, strict=False): name
    for subnet, name in zip(site_subnets_str, cluster_names)
}


def cluster_from_sqlog(sqlog: Path) -> str | None:
    # print(f"sqlog.name: {sqlog.name}")
    match = re.search(r"client-Client-(\d+\.\d+\.\d+\.\d+)\.sqlog", sqlog.name)
    if not match:
        return None

    client_ip = ipaddress.ip_address(match.group(1))
    for subnet, cluster in site_subnet_to_cluster.items():
        if client_ip in subnet:
            return cluster

    return None


# NOTE: for the smoothed RTT extract data.smoothed_rtt from recovery:metrics_updated
def extract_smoothed_rtt(sqlog, cluster, msg_size, writer):
    with open(sqlog) as src:
        for line in src:
            line = line.strip()
            if not line:
                continue

            if '"recovery:metrics_updated"' not in line:
                continue

            try:
                event = orjson.loads(line)
            except ValueError:
                # idk why so many log entries are broken
                continue

            # e.g.
            # {"time":30.884829,"name":"recovery:metrics_updated","data":{"min_rtt":15.767142,"smoothed_rtt":15.767142,"latest_rtt":15.767142,"rtt_variance":7.883571,"bytes_in_flight":0}}
            if event.get("name") == "recovery:metrics_updated":
                smoothed_rtt = event.get("data", {}).get("latest_rtt")
                if smoothed_rtt is not None:
                    writer.writerow(
                        [event.get("time"), cluster, msg_size, smoothed_rtt]
                    )


# to compute the download completion time, take the time from the first stream frame with stream ID 3 and the last, subtract end from start
def extract_download_completion_time(sqlog, cluster, msg_size, writer, run_index):
    with open(sqlog) as src:
        start_frame_time = None
        for line in src:
            line = line.strip()
            if not line:
                continue

            if '"transport:packet_received"' not in line:
                continue

            try:
                event = orjson.loads(line)
            except ValueError:
                # idk why so many log entries are broken
                continue

            # e.g.
            # Small packets: entire message contained in one QUIC packet, contains FIN flag
            #   {"time":5074.468,"name":"transport:packet_received","data":{"header":{"packet_type":"1RTT","packet_number":2},"raw":{"length":1085,"payload_length":1068},"frames":[{"frame_type":"unknown","raw_frame_type":246},{"frame_type":"stream","stream_id":3,"offset":0,"length":1036,"fin":true}]}}
            # large packet:
            # here is one packet that contains part of the message:
            # {"time":5072.8867,"name":"transport:packet_received","data":{"header":{"packet_type":"1RTT","packet_number":3},"raw":{"length":1284,"payload_length":1267},"frames":[{"frame_type":"unknown","raw_frame_type":246},{"frame_type":"stream","stream_id":3,"offset":1235,"length":1234}]}}
            # here is the final packet (with fin flag raised)
            # {"time":5307.8716,"name":"transport:packet_received","data":{"header":{"packet_type":"1RTT","packet_number":92},"raw":{"length":635,"payload_length":618},"frames":[{"frame_type":"unknown","raw_frame_type":246},{"frame_type":"stream","stream_id":3,"offset":99453,"length":583,"fin":true}]}}
            if event.get("name") == "transport:packet_received":
                frames = event.get("data", {}).get("frames", []) or []
                for frame in frames:
                    if (
                        frame.get("frame_type") == "stream"
                        and frame.get("stream_id") == 3
                    ):
                        fin = frame.get("fin", False)
                        # length = frame.get("length")
                        if fin:
                            end_time = event.get("time")
                            if start_frame_time is None:
                                writer.writerow(
                                    [
                                        cluster,
                                        run_index,
                                        msg_size,
                                        end_time,
                                        end_time,
                                        0,
                                    ]
                                )
                                break
                            else:
                                writer.writerow(
                                    [
                                        cluster,
                                        run_index,
                                        msg_size,
                                        start_frame_time,
                                        end_time,
                                        (end_time - start_frame_time),
                                    ]
                                )
                                break
                        else:
                            if start_frame_time is None:
                                start_frame_time = event.get("time")


# Source - https://stackoverflow.com/a/16974075
# Retrieved 2026-09-16, License - CC BY-SA 3.0
def missing_elements(L):
    start, end = L[0], L[-1]
    return sorted(set(range(start, end + 1)).difference(L))


# the quiche version used does not skip packet numbers, so we can store all of the packet numbers we've seen on the FC flow and search for hole to find losses
# NOTE: FCQUIC only sends a SourceSymbol frame (FEC) for packets sent on the FC flow, so here since I can't obtain the path ID of a packet, i'll kinda hack
# my way around it by only storing the packet numbers of the packets that contained a stream frame and a source symbol frame, so that I can know which packets
# from the FC flow i've seen and which were lost
def extract_losses(sqlog, cluster, msg_size, writer, run_id, run_index):
    with open(sqlog) as src:
        packets_seen: set[int] = set()

        for line in src:
            line = line.strip()
            if not line:
                continue

            if '"transport:packet_received"' not in line:
                continue

            try:
                event = orjson.loads(line)
            except ValueError:
                # idk why so many log entries are broken
                continue

            # e.g.
            # {"time":5179.9014,"name":"transport:packet_received","data":{"header":{"packet_type":"1RTT","packet_number":30},"raw":{"length":1284,"payload_length":1267},"frames":[{"frame_type":"unknown","raw_frame_type":246},{"frame_type":"stream","stream_id":3,"offset":31490,"length":1232}]}}
            if event.get("name") == "transport:packet_received":
                data = event.get("data", {})
                frames = data.get("frames", []) or []
                has_source_symbol_frame = False

                for frame in frames:
                    frame_type = frame.get("frame_type")

                    # frames with value 246 (0xF6) are SourceSymbol (FEC) frames
                    if frame_type == "unknown" and frame.get("raw_frame_type") == 246:
                        has_source_symbol_frame = True

                    if (
                        frame_type == "stream"
                        and frame.get("stream_id") == 3
                        and has_source_symbol_frame
                    ) or frame_type == "ping":
                        packet_num = data.get("header").get("packet_number")
                        packets_seen.add(packet_num)

        all_packet_nums = list(packets_seen)
        if len(all_packet_nums) > 0:
            losses = missing_elements(all_packet_nums)
            if len(losses) != 0:
                # print(f"lost packets: {losses}")
                total_lost = len(losses)
                total_sent = len(all_packet_nums)
                loss_rate = total_lost / total_sent
                writer.writerow(
                    [cluster, run_index, msg_size, total_lost, total_sent, loss_rate]
                )


class RowList(list):

    def writerow(self, row):
        self.append(row)


def parse_sqlog_file(trace):
    file, run_index, msg_size = trace
    cluster = cluster_from_sqlog(file)
    if cluster is None:
        return file, None

    rtt, dl, loss = RowList(), RowList(), RowList()
    extract_smoothed_rtt(file, cluster, msg_size, rtt)
    extract_download_completion_time(file, cluster, msg_size, dl, run_index)
    extract_losses(
        file, cluster, msg_size, loss, f"sz{msg_size}_r{run_index}", run_index
    )
    return file, (rtt, dl, loss)


# TODO: remove
# test_name = "categorization_16-09-16-20PM"
# matrix = categorization_matrix()
# N_RUNS=10

local_base = Path(f"./sqlogs/{test_name}")

trace_files = []
for run_conf in matrix:
    for run_index in range(N_RUNS):
        run_id = f"sz{run_conf.additional_data_size}_r{run_index}"

        run_dir = local_base / run_id

        # the qlogs are stored in one dir per client, and the file name contains the client's IP
        for file in sorted(run_dir.glob("client_*/client-*.sqlog")):
            # we keep the message size of the run next to each file so it can be
            # written as a column in the csvs below
            trace_files.append((file, run_index, run_conf.additional_data_size))

if not trace_files:
    print(f"no sqlog files found in {local_base}")


est_rtt_csv = Path(f"./npf-out/") / test_name / f"est_rtt.csv"
dl_completion_csv = Path(f"./npf-out/") / test_name / f"dl_completion.csv"
losses_csv = Path(f"./npf-out/") / test_name / f"losses.csv"
est_rtt_csv.parent.mkdir(parents=True, exist_ok=True)

# the files are parsed in parallel (one worker per file), each row is tagged with the cluster of its client
with (
    open(est_rtt_csv, "w", newline="") as rtt_out,
    open(dl_completion_csv, "w", newline="") as dl_out,
    open(losses_csv, "w", newline="") as losses,
):
    rtt_writer = csv.writer(rtt_out)
    dl_writer = csv.writer(dl_out)
    losses_writer = csv.writer(losses)
    rtt_writer.writerow(["time", "cluster", "msg_size", "smoothed_rtt"])
    dl_writer.writerow(["cluster", "run_index", "msg_size", "start", "end", "duration"])
    losses_writer.writerow(
        ["cluster", "run_index", "msg_size", "lost", "total_msg_packets", "loss_rate"]
    )

    with mp.get_context("fork").Pool() as pool:
        for file, rows in pool.imap(parse_sqlog_file, trace_files):
            if rows is None:
                print(f"couldn't find the cluster of {file}, skipping")
                continue
            rtt_writer.writerows(rows[0])
            dl_writer.writerows(rows[1])
            losses_writer.writerows(rows[2])

print(f"estimated RTT csv: {est_rtt_csv}")
print(f"download completion csv: {dl_completion_csv}")
print(f"losses csv: {losses_csv}")

### Graphing the results


In [ ]:
import subprocess
from pathlib import Path

# test_name="categorization_17-09-16-39PM"

NO_TITLE = True
out_path = f"./graphs/{test_name}"
output_path = Path(out_path)
output_path.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        "./mcast_graphs.py",
        f"./npf-out/raw/{test_name}",
        out_path,  # out path
        test_name,
        f"./npf-out/{test_name}/est_rtt.csv",  # estimated rtt csv
        f"./npf-out/{test_name}/dl_completion.csv",  # download completion time csv
        f"./npf-out/{test_name}/losses.csv",  # losses csv
        *(
            ["--no-title"] if NO_TITLE else []
        ),  # list unpacking, this avoids the empty ""
    ],
    check=True,
)

#### Compressing the csv results

In [ ]:
import subprocess

# compress all related files in one tarball
subprocess.run(
    [
        "tar",
        "czf",
        f"./npf-out/all_{test_name}.tar.gz",
        f"./npf-out/{test_name}.csv",
        f"./npf-out/{test_name}/dl_completion.csv",
        f"./npf-out/{test_name}/est_rtt.csv",
        f"./npf-out/{test_name}/losses.csv",
        f"./npf-out/raw/{test_name}/",
        f"./sqlogs/{test_name}/",
        f"./topology_cache.pkl",
    ],
    check=True,
)

# move archive to the graph dir of the test
subprocess.run(
    [
        "mv",
        f"./npf-out/all_{test_name}.tar.gz",
        f"./graphs/{test_name}/{test_name}.tar.gz",
    ],
    check=True,
)

# delete the csvs and directories
subprocess.run(
    [
        "rm",
        f"./npf-out/{test_name}.csv",
    ],
    check=True,
)
subprocess.run(
    [
        "rm",
        "-rf",
        f"./npf-out/{test_name}/dl_completion.csv",
        f"./npf-out/{test_name}/est_rtt.csv",
        f"./npf-out/{test_name}/losses.csv",
        f"./npf-out/raw/{test_name}/",
        f"./sqlogs/{test_name}/",
    ],
    check=True,
)

Opposite code to unarchive the results, in order to regenerate graphs if needed

In [ ]:
import subprocess
from pathlib import Path

# og_name = "sserv_2thr_latency_test_large_relay_topo_26-04-21-39PM_481mbps"
test_name = "serv_2thr_latency_test_large_relay_topo_26-04-21-39PM"

archive_path = Path(f"./graphs/{test_name}/{test_name}.tar.gz")

if archive_path.exists():
    print(f"decompressing {archive_path}...")

    Path("./npf-out/").mkdir(parents=True, exist_ok=True)

    subprocess.run(
        [
            "tar",
            "xzf",
            str(archive_path),
            "-C",
            "./",
        ],
        check=True,
    )
    print(f"decompressed files to ./npf-out/ and ./sqlogs/")
else:
    print(f"Couldn't find: {archive_path}")

#### Deleting log files from all clusters

In [ ]:
import subprocess
from pathlib import Path

remote_log_root = "/tmp/logs"
matrix = categorization_matrix()
for run_conf in matrix:

    remote_qlog_dir = f"{remote_log_root}/{test_name}/"

    en.run_command(
        f"rm -rf {remote_qlog_dir}",
        roles=experiment.roles["client"] + experiment.roles["server"],
    )

print("done deleting sqlog files")

## Important: Stopping the current booking
Always, always stop your booking if you are done earlier.

In [ ]:
experiment.stop_reservation()